In [5]:
import torch
import torch.nn as nn

class VoiceCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        
       
        self.features = nn.Sequential(
            
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),       
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),     
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

           
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),      
            nn.ReLU(inplace=True),
            
        )

        self.classifier = nn.Sequential(
           
            nn.Dropout(0.0), 
           
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
           
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            
            nn.Dropout(0.0), 
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)




device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VoiceCNN(num_classes=2).to(device)


total_params = sum(p.numel() for p in model.parameters())
print(model)


VoiceCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.0, inplace=False)
    (1): AdaptiveAvgPool2d(output_size=(4, 4))
    (2): Flatten(start_dim=1, end_dim=-1)
    (3): Linear(in_features=1024, out_features=128, bias=Tru

In [7]:
import sounddevice as sd
import numpy as np
import librosa
import torch
import threading
import datetime
from pathlib import Path
from scipy.io.wavfile import write
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, Markdown
from torchvision import transforms
import matplotlib.cm as cm


LABELS = ["accept", "reject"]
SR = 16000
CLIP_SECONDS = 3.0
N_MELS = 128
SILENCE_THRESHOLD = 0.01  
CONFIDENCE_THRESHOLD = 0.70 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


try:
    model = VoiceCNN(num_classes=2).to(device)
    model.load_state_dict(torch.load("final_cnn.pth", map_location=device))
    model.eval()
except NameError:
    print("Error: Model not loaded into memory.")
except FileNotFoundError:
    print("Error: Model file not found.")

save_dir = Path("live_recordings")
save_dir.mkdir(exist_ok=True)
recording = []
is_recording = False



def audio_to_image_tensor(y, sr=16000):
    """
    Generates a spectrogram identical to your matplotlib code.
    - Bass at the TOP (as in your dataset).
    - Dynamic contrast (like plt.imshow default).
    - Stretching to 128x128 (like aspect='auto').
    """
    
    S = librosa.feature.melspectrogram(
        y=y, 
        sr=sr, 
        n_mels=128, 
        n_fft=2048, 
        hop_length=512
    )
    

    S_db = librosa.power_to_db(S, ref=np.max)

    min_val = S_db.min()
    max_val = S_db.max()
   
    if max_val - min_val < 1e-6:
        norm_S = np.zeros_like(S_db)
    else:
        norm_S = (S_db - min_val) / (max_val - min_val)

    cmap = cm.get_cmap("magma")
    img_array = (cmap(norm_S)[:, :, :3] * 255).astype(np.uint8)

   
    img = Image.fromarray(img_array)

    img = img.resize((128, 128), resample=Image.Resampling.NEAREST)
    
    t = transforms.ToTensor()(img).unsqueeze(0).to(device)
    
    return t, img

def is_silence(y, threshold=SILENCE_THRESHOLD):
    return np.sqrt(np.mean(y**2)) < threshold


def record_thread():
    global recording
    recording = []
    def callback(indata, frames, time, status):
        if is_recording: recording.append(indata.copy())
    with sd.InputStream(samplerate=SR, channels=1, callback=callback):
        while is_recording: sd.sleep(100)

output = widgets.Output()
button = widgets.Button(description="Start Recording", button_style="success", icon='microphone')

def on_click(b):
    global is_recording
    if not is_recording:
        is_recording = True
        b.description = "Stop Recording"
        b.button_style = "danger"
        threading.Thread(target=record_thread, daemon=True).start()
    else:
        is_recording = False
        b.description = "Start Recording"
        b.button_style = "success"
        
        if not recording: return

        y_raw = np.concatenate(recording).ravel()

        if is_silence(y_raw):
            with output:
                output.clear_output()
                print("Too quiet (Silence)")
            return

        y, _ = librosa.effects.trim(y_raw, top_db=20)
        needed = int(CLIP_SECONDS * SR)
        if len(y) < needed: 
          
            y = np.pad(y, (0, needed - len(y)), mode='constant') 
        else: 
            y = y[:needed]

        tensor_in, pil_img = audio_to_image_tensor(y, SR)
        
        with torch.no_grad():
            out = model(tensor_in)
            probs = torch.softmax(out, dim=1)[0]
            pred_idx = probs.argmax().item()
            conf = probs[pred_idx].item()

        label = LABELS[pred_idx]
        color = "green" if label == "accept" and conf > CONFIDENCE_THRESHOLD else "red"
        
        if conf < CONFIDENCE_THRESHOLD:
            label = f"{label} (Uncertain)"

        with output:
            output.clear_output()
            display(pil_img)
            display(Markdown(f"### <span style='color:{color}'>{label}</span>"))
            display(Markdown(f"Confidence: **{conf:.1%}**"))

button.on_click(on_click)
display(button, output)

Button(button_style='success', description='Start Recording', icon='microphone', style=ButtonStyle())

Output()